# 01: NIH ChestX-ray14 data on Colab

**For education only, not for clinical use.**

Downloads the official NIH dataset to the Colab VM, builds the patient-level manifest, runs EDA and
writes 224x224 grayscale shards to Drive.

- Use a **CPU runtime**: nothing here needs a GPU, and CPU runtimes have more disk. The zip needs about
  45 GB of VM disk. It is read directly, never extracted.
- **Raw data never goes to Drive.** Only the manifest, reports and shards do.
- Before running, add `KAGGLE_USERNAME` and `KAGGLE_KEY` as Colab secrets, or upload `kaggle.json` when
  asked. Credentials stay on the VM.
- If the runtime disconnects, run all cells again: finished shards are kept and skipped.
- The shard step stops by itself if the projected total reaches 4 GB.

## 1. Drive and paths

In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/medical-imaging-ai")
MANIFESTS = DRIVE_ROOT / "manifests"
REPORTS = DRIVE_ROOT / "reports"
SHARDS = DRIVE_ROOT / "shards_224"
for path in (MANIFESTS, REPORTS, SHARDS):
    path.mkdir(parents=True, exist_ok=True)

# Raw data and temporary files stay on the VM.
RAW = Path("/content/raw")
META = Path("/content/meta")
WORK = Path("/content/work")
for path in (RAW, META, WORK):
    assert not str(path).startswith("/content/drive"), f"{path} must not be on Drive"
    path.mkdir(parents=True, exist_ok=True)

free_gb = shutil.disk_usage("/content").free / 1e9
print(f"VM disk free: {free_gb:.1f} GB")
already_downloaded = any(RAW.glob("*.zip"))
assert already_downloaded or free_gb > 50, "Need about 50 GB free; switch to a CPU runtime"

## 2. Code from the repo

In [ ]:
import subprocess
import sys

REPO_URL = "https://github.com/manojkumargudge/medical-imaging-ai.git"
REPO_REF = "nih-data"  # Change to "main" once this branch is merged.
REPO = Path("/content/medical-imaging-ai")

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)], check=True
    )
# The repo is on the path rather than installed: its pins are for local Python 3.11, and Colab's
# own numpy, pandas and pillow are enough here.
sys.path.insert(0, str(REPO))
print(subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout)

from src.data import nih, shards

## 3. Kaggle credentials

In [ ]:
import json

try:
    from google.colab import userdata

    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Using Colab secrets")
except Exception as exc:
    print(f"No Colab secrets ({type(exc).__name__}); upload kaggle.json")
    from google.colab import files

    uploaded = files.upload()
    credentials = json.loads(uploaded["kaggle.json"])
    os.environ["KAGGLE_USERNAME"] = credentials["username"]
    os.environ["KAGGLE_KEY"] = credentials["key"]
    # Removes the uploaded copy; the key stays only in this session's environment.
    Path("kaggle.json").unlink(missing_ok=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)

## 4. Verify the dataset slug

In [ ]:
SLUG = "nih-chest-xrays/data"

listing = subprocess.run(
    ["kaggle", "datasets", "files", SLUG], capture_output=True, text=True, check=True
)
print("\n".join(listing.stdout.splitlines()[:15]))
assert any(ext in listing.stdout for ext in (".png", ".csv", ".txt")), (
    f"No dataset files listed for {SLUG}: {listing.stdout[:500]} {listing.stderr[:500]}"
)

## 5. Download to the VM (about 45 GB, not extracted)

In [ ]:
if not any(RAW.glob("*.zip")):
    subprocess.run(["kaggle", "datasets", "download", SLUG, "-p", str(RAW)], check=True)

zips = list(RAW.glob("*.zip"))
assert len(zips) == 1, f"Expected one zip, found {zips}"
print(zips[0], f"{zips[0].stat().st_size / 1e9:.1f} GB")

## 6. Check the image count and get the metadata files

In [ ]:
images = shards.ZipImages(zips[0])
print(f"PNG images in zip: {len(images.names)}")
assert len(images.names) == nih.EXPECTED_IMAGES, f"Expected {nih.EXPECTED_IMAGES} images"

data_entry = images.extract_file("Data_Entry_2017.csv", META)
train_val_list = images.extract_file("train_val_list.txt", META)
test_list = images.extract_file("test_list.txt", META)

## 7. Build the manifest with `src/data/nih.py`

In [ ]:
import pandas as pd

manifest_path = WORK / "nih_manifest.csv"
subprocess.run(
    [
        sys.executable, "-m", "src.data.nih",
        "--data-entry", str(data_entry),
        "--train-val-list", str(train_val_list),
        "--test-list", str(test_list),
        "--out", str(manifest_path),
        "--seed", "42",
    ],
    check=True,
    cwd=REPO,
)
manifest = pd.read_csv(manifest_path, dtype={"patient": str, "image": str})
nih.assert_patient_level_splits(manifest)

missing = set(manifest["image"]) - images.names
extra = images.names - set(manifest["image"])
print(f"manifest rows {len(manifest)}, zip images {len(images.names)}, "
      f"missing from zip {len(missing)}, not in manifest {len(extra)}")
assert len(manifest) > 0 and len(images.names) > 0
assert not missing and not extra

shutil.copyfile(manifest_path, MANIFESTS / manifest_path.name)

## 8. EDA: label prevalence, age/sex, images per patient

In [ ]:
import matplotlib.pyplot as plt

for path in nih.write_eda_tables(manifest, REPORTS):
    print(path)

prevalence = nih.label_prevalence(manifest)
display(prevalence)
demographics = nih.age_sex_by_split(manifest)
display(demographics)
per_patient = nih.images_per_patient(manifest)
display(per_patient)

fig, ax = plt.subplots(figsize=(8, 5))
prevalence["all_percent"].sort_values().plot.barh(ax=ax)
ax.set_xlabel("Images with label (%)")
ax.set_ylabel("")
ax.set_title("Label prevalence, all splits")
fig.tight_layout()
fig.savefig(REPORTS / "label_prevalence.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
for sex, rows in manifest.groupby("sex"):
    ax.hist(rows["age"].dropna(), bins=range(0, 105, 5), alpha=0.6, label=sex)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Images")
ax.set_title("Age by sex, per image")
ax.legend(title="Sex")
fig.tight_layout()
fig.savefig(REPORTS / "age_by_sex.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
counts = manifest.groupby("patient").size()
ax.hist(counts, bins=50, log=True)
ax.set_xlabel("Images per patient")
ax.set_ylabel("Patients (log scale)")
ax.set_title("Images per patient")
fig.tight_layout()
fig.savefig(REPORTS / "images_per_patient.png", dpi=150)
plt.show()

## 9. Resize to 224 grayscale and write shards to Drive

Uncompressed this would be 5.6 GB, so the 4 GB limit depends on compression. The size is projected
after every shard. If the run stops on the limit, keep the finished shards and decide what to do
before running it again.

In [ ]:
report = shards.write_shards(manifest, images.read, SHARDS, WORK / "shards")
images.close()

stored = sorted(SHARDS.glob("*.npz"))
stored_bytes = sum(path.stat().st_size for path in stored)
print(f"manifest rows {len(manifest)}, images in shards {report.images}")
print(f"shards {len(stored)}, size on Drive {stored_bytes / 1e9:.2f} GB")
assert report.images == len(manifest) > 0
assert stored_bytes < shards.MAX_TOTAL_BYTES, "Shards must stay under 4 GB"
assert not list(SHARDS.glob("*.partial"))

(REPORTS / "shards_report.json").write_text(
    json.dumps(
        {
            "images": report.images,
            "shards": len(stored),
            "total_bytes": stored_bytes,
            "image_size": shards.IMAGE_SIZE,
            "encoding": shards.ENCODING,
        },
        indent=2,
    )
)

# Spot check: first image of the first shard, read back.
sample_images, sample_names = shards.read_shard(stored[0])
plt.imshow(sample_images[0], cmap="gray")
plt.title(sample_names[0])
plt.axis("off")
plt.show()

## 10. Afterwards, on the local machine

1. Download `MyDrive/medical-imaging-ai/manifests/nih_manifest.csv` to `data/manifests/`.
2. Download `MyDrive/medical-imaging-ai/reports/` to `reports/`.
3. Version and commit:

```powershell
.venv\Scripts\dvc add data/manifests
.venv\Scripts\dvc push
git add data/manifests.dvc reports
git commit -m "Add NIH manifest and EDA reports"
git push
```